In [1]:
# 1. Import libraries and load cleaned data

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

# Load cleaned Understat shot data
DATA_PATH = "../data/cleaned/shots_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Number of missing values: {df.isnull().sum().sum()}")

df.head()


Dataset shape: (50798, 38)
Number of missing values: 0


,is_home,team,season,xg,minute,result,x,y,angle_to_goal,is_assisted,...,last_action_foul,last_action_head_pass,last_action_lay_off,last_action_none,last_action_pass,last_action_rebound,last_action_standard,last_action_tackle,last_action_take_on,last_action_throughball
0,1,Brentford,2021/22,0.053464,10,0,109.560004,43.120001,0.714539,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Brentford,2021/22,0.118297,11,0,108.960004,25.200000,0.288597,1,...,0,0,0,0,0,0,0,0,0,1
2,1,Brentford,2021/22,0.052145,21,1,104.880002,55.840002,0.275243,1,...,0,0,0,0,0,0,0,0,0,0
3,1,Brentford,2021/22,0.065706,27,0,97.439996,38.239999,0.364677,1,...,0,0,0,0,1,0,0,0,0,0
4,1,Brentford,2021/22,0.080691,29,0,107.039996,28.560001,0.376828,1,...,0,0,0,0,0,0,0,0,0,0


In [2]:

# 2. Define the agreed season split

TRAIN_SEASONS = ["2021/22", "2022/23", "2023/24"]
VAL_SEASONS   = ["2024/25"]
TEST_SEASONS  = ["2025/26"]  # Held out until decisions are frozen

# 2026/27 is excluded because only 60 matches are available so far

# Keep only the agreed seasons
allowed_seasons = TRAIN_SEASONS + VAL_SEASONS + TEST_SEASONS

df = df[df["season"].isin(allowed_seasons)].copy()

# Create chronological train, validation and test datasets
train_df = df[df["season"].isin(TRAIN_SEASONS)].copy()
val_df   = df[df["season"].isin(VAL_SEASONS)].copy()
test_df  = df[df["season"].isin(TEST_SEASONS)].copy()

# Verify the split
print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

print("\nTraining seasons:")
print(sorted(train_df["season"].unique()))

print("\nValidation seasons:")
print(sorted(val_df["season"].unique()))

print("\nTest seasons:")
print(sorted(test_df["season"].unique()))


Training set: (29813, 38)
Validation set: (9844, 38)
Test set: (9483, 38)

Training seasons:
['2021/22', '2022/23', '2023/24']

Validation seasons:
['2024/25']

Test seasons:
['2025/26']


In [3]:

# 4. Prepare datasets, define predictors and resolve rank deficiency

from scipy.linalg import qr

# Convert Understat coordinates to metres and create spatial features
PITCH_L, PITCH_W = 105, 68

def add_spatial_features(data):
    data = data.copy()

    x_m = data["x"] * PITCH_L / 120
    y_m = data["y"] * PITCH_W / 80

    data["dist_goal_line"] = PITCH_L - x_m
    data["lateral_offset"] = (y_m - PITCH_W / 2).abs()

    return data


# Create the datasets used by the models
train = add_spatial_features(train_df)
val = add_spatial_features(val_df)
test = add_spatial_features(test_df)


# Define predictor groups
CORE_FEATURES = [
    "dist_goal_line",
    "lateral_offset",
    "angle_to_goal"
]

MATCH_FEATURES = [
    "is_home",
    "minute",
    "score_diff"
]

SHOT_FEATURES = [
    "is_assisted",
    "is_penalty",
    "is_open_play",
    "from_corner",
    "is_free_kick"
]

BODY_FEATURES = [
    "left_foot",
    "other_body_part",
    "right_foot"
]

# Exclude categories previously found to cause separation
LAST_ACTION_FEATURES = [
    col for col in train.columns
    if col.startswith("last_action_")
    and col not in ["last_action_none", "last_action_foul"]
]

FULL_FEATURES = (
    CORE_FEATURES
    + MATCH_FEATURES
    + SHOT_FEATURES
    + BODY_FEATURES
    + LAST_ACTION_FEATURES
)

# Check that all specified predictors exist
missing_features = [
    col for col in FULL_FEATURES
    if col not in train.columns
]

if missing_features:
    raise ValueError(
        f"Predictors missing from training data: {missing_features}"
    )

# Check for missing or infinite predictor values
X_check = sm.add_constant(
    train[FULL_FEATURES],
    has_constant="add"
)

if not np.isfinite(X_check.to_numpy()).all():
    raise ValueError(
        "The design matrix contains missing or infinite values."
    )

# Identify linearly dependent columns
X_array = X_check.to_numpy()

_, _, pivot = qr(
    X_array,
    mode="economic",
    pivoting=True
)

rank = np.linalg.matrix_rank(X_array)

independent_cols = [
    X_check.columns[i] for i in pivot[:rank]
]

redundant_cols = [
    col for col in X_check.columns
    if col not in independent_cols
]

print("Original number of columns:", X_check.shape[1])
print("Original matrix rank:", rank)
print("Redundant columns identified:", redundant_cols)

# Remove redundant predictors, keeping the intercept
if "const" in redundant_cols:
    raise ValueError(
        "The intercept was identified as redundant. "
        "Inspect the design matrix."
    )

FULL_FEATURES = [
    col for col in FULL_FEATURES
    if col not in redundant_cols
]

# Verify that the design matrix now has full rank
X_check = sm.add_constant(
    train[FULL_FEATURES],
    has_constant="add"
)

new_rank = np.linalg.matrix_rank(X_check.to_numpy())

print("\nUpdated number of columns:", X_check.shape[1])
print("Updated matrix rank:", new_rank)
print("Removed predictors:", redundant_cols)

assert new_rank == X_check.shape[1], (
    "The design matrix is still rank-deficient."
)

print("\nSuccess: datasets and predictors are ready.")


Original number of columns: 29
Original matrix rank: 28
Redundant columns identified: ['is_free_kick']

Updated number of columns: 28
Updated matrix rank: 28
Removed predictors: ['is_free_kick']

Success: datasets and predictors are ready.


In [4]:

# 5. Fit the full logistic regression model

X_train_full = sm.add_constant(
    train[FULL_FEATURES],
    has_constant="add"
)

y_train = train["result"].to_numpy()

full_fit = sm.Logit(
    y_train,
    X_train_full
).fit(method="lbfgs", maxiter=1000)

print(full_fit.summary())
print("\nConverged:", full_fit.mle_retvals["converged"])


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =           28     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.93147D-01    |proj g|=  1.88477D+01

At iterate    1    f=  4.89100D-01    |proj g|=  4.22737D+00

At iterate    2    f=  4.39995D-01    |proj g|=  3.59864D+00

At iterate    3    f=  3.78370D-01    |proj g|=  1.43259D+00

At iterate    4    f=  3.63823D-01    |proj g|=  1.04063D+00

At iterate    5    f=  3.52397D-01    |proj g|=  1.10438D+00

At iterate    6    f=  3.24472D-01    |proj g|=  1.16503D+00

At iterate    7    f=  3.15309D-01    |proj g|=  1.53850D-01

At iterate    8    f=  3.13283D-01    |proj g|=  8.75863D-02

At iterate    9    f=  3.12322D-01    |proj g|=  7.54453D-02

At iterate   10    f=  3.09989D-01    |proj g|=  6.11176D-02

At iterate   11    f=  3.09211D-01    |proj g|=  3.06904D-02

At iterate   12    f=  3.08780D-01    |proj g|=  1.87649D-02

At iterate   13    f=  3.0

 This problem is unconstrained.



At iterate   85    f=  2.84046D-01    |proj g|=  2.52664D-03

At iterate   86    f=  2.84044D-01    |proj g|=  1.10232D-02

At iterate   87    f=  2.84041D-01    |proj g|=  7.28401D-03

At iterate   88    f=  2.84030D-01    |proj g|=  2.18097D-02

At iterate   89    f=  2.84015D-01    |proj g|=  2.66282D-02

At iterate   90    f=  2.83990D-01    |proj g|=  2.33612D-02

At iterate   91    f=  2.83761D-01    |proj g|=  2.25074D-02

At iterate   92    f=  2.83575D-01    |proj g|=  6.76390D-02

At iterate   93    f=  2.83241D-01    |proj g|=  4.71194D-02

At iterate   94    f=  2.83125D-01    |proj g|=  1.35559D-01

At iterate   95    f=  2.82898D-01    |proj g|=  5.62975D-02

At iterate   96    f=  2.82754D-01    |proj g|=  2.55210D-02

At iterate   97    f=  2.82747D-01    |proj g|=  1.06358D-02

At iterate   98    f=  2.82742D-01    |proj g|=  2.88937D-03

At iterate   99    f=  2.82736D-01    |proj g|=  1.33958D-02

At iterate  100    f=  2.82727D-01    |proj g|=  1.44306D-02

At iter


At iterate  218    f=  2.80574D-01    |proj g|=  4.24699D-03

At iterate  219    f=  2.80569D-01    |proj g|=  2.83246D-03

At iterate  220    f=  2.80567D-01    |proj g|=  3.25918D-03

At iterate  221    f=  2.80566D-01    |proj g|=  3.61253D-03

At iterate  222    f=  2.80564D-01    |proj g|=  8.23864D-03

At iterate  223    f=  2.80563D-01    |proj g|=  1.79366D-03

At iterate  224    f=  2.80562D-01    |proj g|=  3.40701D-03

At iterate  225    f=  2.80561D-01    |proj g|=  1.47768D-03

At iterate  226    f=  2.80561D-01    |proj g|=  9.69879D-03

At iterate  227    f=  2.80560D-01    |proj g|=  2.27059D-03

At iterate  228    f=  2.80559D-01    |proj g|=  2.39822D-03

At iterate  229    f=  2.80558D-01    |proj g|=  5.41406D-03

At iterate  230    f=  2.80556D-01    |proj g|=  5.71430D-03

At iterate  231    f=  2.80554D-01    |proj g|=  6.50110D-03

At iterate  232    f=  2.80553D-01    |proj g|=  1.68131D-02

At iterate  233    f=  2.80551D-01    |proj g|=  7.54844D-03

At iter


At iterate  380    f=  2.80254D-01    |proj g|=  4.82022D-04

At iterate  381    f=  2.80254D-01    |proj g|=  3.18965D-04

At iterate  382    f=  2.80253D-01    |proj g|=  2.19015D-03

At iterate  383    f=  2.80253D-01    |proj g|=  5.50903D-04

At iterate  384    f=  2.80253D-01    |proj g|=  6.24824D-03

At iterate  385    f=  2.80253D-01    |proj g|=  1.81591D-03

At iterate  386    f=  2.80253D-01    |proj g|=  6.26426D-04

At iterate  387    f=  2.80253D-01    |proj g|=  3.39715D-03

At iterate  388    f=  2.80252D-01    |proj g|=  6.01511D-03

At iterate  389    f=  2.80252D-01    |proj g|=  7.15192D-03

At iterate  390    f=  2.80251D-01    |proj g|=  6.63297D-03

At iterate  391    f=  2.80251D-01    |proj g|=  3.30791D-03

At iterate  392    f=  2.80251D-01    |proj g|=  4.20431D-04

At iterate  393    f=  2.80250D-01    |proj g|=  4.21173D-03

At iterate  394    f=  2.80250D-01    |proj g|=  6.00495D-03

At iterate  395    f=  2.80250D-01    |proj g|=  8.92917D-03

At iter


At iterate  543    f=  2.80177D-01    |proj g|=  2.90917D-03

At iterate  544    f=  2.80177D-01    |proj g|=  4.50962D-03

At iterate  545    f=  2.80176D-01    |proj g|=  3.40126D-03

At iterate  546    f=  2.80176D-01    |proj g|=  7.63828D-03

At iterate  547    f=  2.80175D-01    |proj g|=  5.49343D-03

At iterate  548    f=  2.80175D-01    |proj g|=  2.40713D-03

At iterate  549    f=  2.80175D-01    |proj g|=  4.59645D-04

At iterate  550    f=  2.80175D-01    |proj g|=  1.26228D-04

At iterate  551    f=  2.80175D-01    |proj g|=  1.07919D-03

At iterate  552    f=  2.80175D-01    |proj g|=  1.83537D-03

At iterate  553    f=  2.80174D-01    |proj g|=  2.21177D-03

At iterate  554    f=  2.80174D-01    |proj g|=  4.45803D-04

At iterate  555    f=  2.80174D-01    |proj g|=  4.53457D-04

At iterate  556    f=  2.80174D-01    |proj g|=  3.68441D-03

At iterate  557    f=  2.80174D-01    |proj g|=  8.01233D-04

At iterate  558    f=  2.80174D-01    |proj g|=  4.89422D-04

At iter


At iterate  706    f=  2.80161D-01    |proj g|=  1.30786D-03

At iterate  707    f=  2.80161D-01    |proj g|=  1.04056D-03

At iterate  708    f=  2.80161D-01    |proj g|=  6.56258D-04

At iterate  709    f=  2.80161D-01    |proj g|=  1.75616D-03

At iterate  710    f=  2.80161D-01    |proj g|=  1.86225D-03

At iterate  711    f=  2.80161D-01    |proj g|=  3.67252D-04

At iterate  712    f=  2.80161D-01    |proj g|=  5.44313D-04

At iterate  713    f=  2.80161D-01    |proj g|=  3.25400D-03

At iterate  714    f=  2.80161D-01    |proj g|=  2.23050D-03

At iterate  715    f=  2.80161D-01    |proj g|=  6.94945D-04

At iterate  716    f=  2.80161D-01    |proj g|=  5.91068D-04

At iterate  717    f=  2.80160D-01    |proj g|=  1.57544D-03

At iterate  718    f=  2.80160D-01    |proj g|=  1.59591D-03

At iterate  719    f=  2.80160D-01    |proj g|=  2.22012D-03

At iterate  720    f=  2.80160D-01    |proj g|=  3.17065D-04

At iterate  721    f=  2.80160D-01    |proj g|=  1.53315D-04

At iter

In [5]:

# 5. Fit the full logistic regression model

X_train_full = sm.add_constant(
    train[FULL_FEATURES],
    has_constant="add"
)

y_train = train["result"].to_numpy()

full_fit = sm.Logit(
    y_train,
    X_train_full
).fit(method="lbfgs", maxiter=1000)

print(full_fit.summary())
print("\nConverged:", full_fit.mle_retvals["converged"])

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =           28     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.93147D-01    |proj g|=  1.88477D+01

At iterate    1    f=  4.89100D-01    |proj g|=  4.22737D+00

At iterate    2    f=  4.39995D-01    |proj g|=  3.59864D+00

At iterate    3    f=  3.78370D-01    |proj g|=  1.43259D+00

At iterate    4    f=  3.63823D-01    |proj g|=  1.04063D+00

At iterate    5    f=  3.52397D-01    |proj g|=  1.10438D+00

At iterate    6    f=  3.24472D-01    |proj g|=  1.16503D+00

At iterate    7    f=  3.15309D-01    |proj g|=  1.53850D-01

At iterate    8    f=  3.13283D-01    |proj g|=  8.75863D-02

At iterate    9    f=  3.12322D-01    |proj g|=  7.54453D-02

At iterate   10    f=  3.09989D-01    |proj g|=  6.11176D-02

At iterate   11    f=  3.09211D-01    |proj g|=  3.06904D-02

At iterate   12    f=  3.08780D-01    |proj g|=  1.87649D-02

At iterate   13    f=  3.0

 This problem is unconstrained.



At iterate   54    f=  2.87054D-01    |proj g|=  1.47032D-02

At iterate   55    f=  2.87040D-01    |proj g|=  2.45042D-02

At iterate   56    f=  2.86937D-01    |proj g|=  6.45501D-02

At iterate   57    f=  2.86802D-01    |proj g|=  5.43327D-02

At iterate   58    f=  2.86636D-01    |proj g|=  3.71218D-02

At iterate   59    f=  2.86557D-01    |proj g|=  1.45792D-02

At iterate   60    f=  2.86492D-01    |proj g|=  1.27802D-02

At iterate   61    f=  2.86427D-01    |proj g|=  7.40334D-02

At iterate   62    f=  2.86319D-01    |proj g|=  4.14816D-02

At iterate   63    f=  2.86123D-01    |proj g|=  1.58708D-01

At iterate   64    f=  2.85943D-01    |proj g|=  2.81655D-01

At iterate   65    f=  2.85567D-01    |proj g|=  7.54326D-02

At iterate   66    f=  2.85268D-01    |proj g|=  4.54750D-02

At iterate   67    f=  2.85204D-01    |proj g|=  2.91606D-02

At iterate   68    f=  2.85200D-01    |proj g|=  3.13646D-02

At iterate   69    f=  2.85177D-01    |proj g|=  4.15713D-02

At iter


At iterate  202    f=  2.80697D-01    |proj g|=  2.52344D-02

At iterate  203    f=  2.80694D-01    |proj g|=  1.78175D-03

At iterate  204    f=  2.80694D-01    |proj g|=  1.73477D-03

At iterate  205    f=  2.80693D-01    |proj g|=  2.73154D-03

At iterate  206    f=  2.80690D-01    |proj g|=  7.84204D-03

At iterate  207    f=  2.80686D-01    |proj g|=  1.18371D-02

At iterate  208    f=  2.80679D-01    |proj g|=  1.08712D-02

At iterate  209    f=  2.80670D-01    |proj g|=  2.11785D-02

At iterate  210    f=  2.80665D-01    |proj g|=  7.57501D-03

At iterate  211    f=  2.80655D-01    |proj g|=  4.94135D-03

At iterate  212    f=  2.80632D-01    |proj g|=  1.14879D-02

At iterate  213    f=  2.80616D-01    |proj g|=  9.37312D-03

At iterate  214    f=  2.80615D-01    |proj g|=  2.65748D-02

At iterate  215    f=  2.80596D-01    |proj g|=  1.22345D-02

At iterate  216    f=  2.80586D-01    |proj g|=  7.51924D-03

At iterate  217    f=  2.80579D-01    |proj g|=  5.48324D-03

At iter


At iterate  347    f=  2.80282D-01    |proj g|=  2.05886D-02

At iterate  348    f=  2.80279D-01    |proj g|=  6.65263D-03

At iterate  349    f=  2.80275D-01    |proj g|=  5.08154D-03

At iterate  350    f=  2.80274D-01    |proj g|=  2.33714D-03

At iterate  351    f=  2.80274D-01    |proj g|=  7.18279D-03

At iterate  352    f=  2.80273D-01    |proj g|=  1.09372D-03

At iterate  353    f=  2.80273D-01    |proj g|=  3.16856D-03

At iterate  354    f=  2.80273D-01    |proj g|=  4.72215D-03

At iterate  355    f=  2.80272D-01    |proj g|=  1.40747D-03

At iterate  356    f=  2.80272D-01    |proj g|=  6.11952D-04

At iterate  357    f=  2.80272D-01    |proj g|=  2.74110D-03

At iterate  358    f=  2.80272D-01    |proj g|=  5.30828D-03

At iterate  359    f=  2.80272D-01    |proj g|=  5.57751D-03

At iterate  360    f=  2.80272D-01    |proj g|=  7.32114D-03

At iterate  361    f=  2.80272D-01    |proj g|=  3.12611D-03

At iterate  362    f=  2.80271D-01    |proj g|=  1.47514D-03

At iter


At iterate  482    f=  2.80189D-01    |proj g|=  1.83495D-03

At iterate  483    f=  2.80189D-01    |proj g|=  4.77648D-03

At iterate  484    f=  2.80189D-01    |proj g|=  1.22777D-03

At iterate  485    f=  2.80189D-01    |proj g|=  8.18661D-04

At iterate  486    f=  2.80188D-01    |proj g|=  3.23830D-03

At iterate  487    f=  2.80188D-01    |proj g|=  2.48662D-03

At iterate  488    f=  2.80188D-01    |proj g|=  3.15370D-03

At iterate  489    f=  2.80188D-01    |proj g|=  1.01700D-03

At iterate  490    f=  2.80188D-01    |proj g|=  1.91699D-03

At iterate  491    f=  2.80188D-01    |proj g|=  8.43055D-03

At iterate  492    f=  2.80188D-01    |proj g|=  5.16838D-03

At iterate  493    f=  2.80187D-01    |proj g|=  4.45449D-04

At iterate  494    f=  2.80187D-01    |proj g|=  6.29126D-03

At iterate  495    f=  2.80186D-01    |proj g|=  8.71188D-03

At iterate  496    f=  2.80185D-01    |proj g|=  3.98011D-03

At iterate  497    f=  2.80185D-01    |proj g|=  5.24970D-04

At iter


At iterate  622    f=  2.80167D-01    |proj g|=  3.40439D-04

At iterate  623    f=  2.80167D-01    |proj g|=  4.54859D-04

At iterate  624    f=  2.80167D-01    |proj g|=  1.39745D-03

At iterate  625    f=  2.80167D-01    |proj g|=  3.07120D-04

At iterate  626    f=  2.80167D-01    |proj g|=  1.68288D-04

At iterate  627    f=  2.80167D-01    |proj g|=  1.78404D-04

At iterate  628    f=  2.80167D-01    |proj g|=  5.49462D-04

At iterate  629    f=  2.80167D-01    |proj g|=  7.68850D-05

At iterate  630    f=  2.80167D-01    |proj g|=  8.69459D-04

At iterate  631    f=  2.80167D-01    |proj g|=  2.60713D-04

At iterate  632    f=  2.80167D-01    |proj g|=  4.68934D-04

At iterate  633    f=  2.80167D-01    |proj g|=  8.75903D-04

At iterate  634    f=  2.80167D-01    |proj g|=  1.11579D-03

At iterate  635    f=  2.80167D-01    |proj g|=  3.67402D-04

At iterate  636    f=  2.80167D-01    |proj g|=  2.02096D-03

At iterate  637    f=  2.80167D-01    |proj g|=  2.87354D-04

At iter

In [6]:

# Diagnose rare categories and possible predictor redundancy

print("=== CATEGORY COUNTS ===")

for col in SHOT_FEATURES + BODY_FEATURES + LAST_ACTION_FEATURES:
    if col in train.columns:
        print(f"\n{col}")
        print(train[col].value_counts(dropna=False).head(10))

print("\n=== POSSIBLE REDUNDANCY ===")

# Check whether any binary predictors are identical or exact opposites
binary_features = [
    col for col in FULL_FEATURES
    if train[col].nunique() <= 2
]

for i, col1 in enumerate(binary_features):
    for col2 in binary_features[i + 1:]:
        a = train[col1].to_numpy()
        b = train[col2].to_numpy()

        if np.array_equal(a, b):
            print(f"Identical predictors: {col1} and {col2}")
        elif np.array_equal(a, 1 - b):
            print(f"Opposite predictors: {col1} and {col2}")

print("\n=== GOAL COUNTS IN CATEGORIES ===")

for col in SHOT_FEATURES + BODY_FEATURES + LAST_ACTION_FEATURES:
    if col in train.columns:
        print(f"\n{col}")
        print(pd.crosstab(train[col], train["result"]))


=== CATEGORY COUNTS ===

is_assisted
1    22066
0     7747
Name: is_assisted, dtype: int64

is_penalty
0    29504
1      309
Name: is_penalty, dtype: int64

is_open_play
1    21724
0     8089
Name: is_open_play, dtype: int64

from_corner
0    24743
1     5070
Name: from_corner, dtype: int64

is_free_kick
0    28894
1      919
Name: is_free_kick, dtype: int64

left_foot
0    20129
1     9684
Name: left_foot, dtype: int64

other_body_part
0    29680
1      133
Name: other_body_part, dtype: int64

right_foot
1    14968
0    14845
Name: right_foot, dtype: int64

last_action_aerial
0    27349
1     2464
Name: last_action_aerial, dtype: int64

last_action_ball_recovery
0    29051
1      762
Name: last_action_ball_recovery, dtype: int64

last_action_ball_touch
0    29166
1      647
Name: last_action_ball_touch, dtype: int64

last_action_chipped
0    28313
1     1500
Name: last_action_chipped, dtype: int64

last_action_cross
0    26075
1     3738
Name: last_action_cross, dtype: int64

last_act

In [7]:
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

# Core model: spatial predictors only
X_train_core = sm.add_constant(
    train[CORE_FEATURES],
    has_constant="add"
)
X_val_core = sm.add_constant(
    val[CORE_FEATURES],
    has_constant="add"
)

core_fit = sm.Logit(
    train["result"],
    X_train_core
).fit(disp=False)

# Full model: all selected predictors
X_train_full = sm.add_constant(
    train[FULL_FEATURES],
    has_constant="add"
)
X_val_full = sm.add_constant(
    val[FULL_FEATURES],
    has_constant="add"
)

full_fit = sm.Logit(
    train["result"],
    X_train_full
).fit(method="lbfgs", maxiter=1000, disp=False)

# Validation predictions
y_val = val["result"].to_numpy()

p_core = np.asarray(core_fit.predict(X_val_core))
p_full = np.asarray(full_fit.predict(X_val_full))

# Compare out-of-sample performance
results = pd.DataFrame({
    "Model": ["Core", "Full"],
    "ROC_AUC": [
        roc_auc_score(y_val, p_core),
        roc_auc_score(y_val, p_full)
    ],
    "Brier_Score": [
        brier_score_loss(y_val, p_core),
        brier_score_loss(y_val, p_full)
    ],
    "Log_Loss": [
        log_loss(y_val, p_core),
        log_loss(y_val, p_full)
    ]
})

results

,Model,ROC_AUC,Brier_Score,Log_Loss
0,Core,0.718191,0.089532,0.313183
1,Full,0.780692,0.081090,0.285322
